# HHBANK - 2 COUNTRY

In [ ]:
# PACKAGES AND PATHS
import numpy as np
import matplotlib.pyplot as plt
import sequence_jacobian as sj
from sequence_jacobian import simple, solved, combine, create_model
from sequence_jacobian import grids, hetblocks

from pathlib import Path
import numpy as np


from pathlib import Path
import numpy as np
from sequence_jacobian import grids

# 1. Robust Path Definition
try:
    # Works if running as a .py script
    BASE_DIR_D = Path(__file__).resolve().parent
except NameError:
    # Works if running in a Jupyter Notebook
    BASE_DIR_D = Path.cwd()

# REMOVE .parent here. cwc
# Your image shows 'Discretisation' is in the same folder as the notebook.
DATA_DIR_D = BASE_DIR_D / "Discretisation" / "Outputs"

In [ ]:
# CALIBRATION
calibration_start = {

    #==> Household parameters
    'frisch_D': 1.0,    'frisch_F': 1.0,
    'eis_D': 0.5,       'eis_F': 0.5,
    'habit_D': 0.00,    'habit_F': 0.00,
    'C_lag_D': 0.0,     'C_lag_F': 0.0,

    #==> Deposit rate (= real rate at SS since pi=0)
    'rdep_D': 0.0065,   'rdep_F': 0.0062,

    #==> Bond rate (initial guess; SS value will be solved endogenously via interest_rates_D/F)
    'rb_D': 0.0099,     'rb_F': 0.0075,

    #==> Government bonds
    'B_supply_D': 0.6 * 4,  'B_supply_F': 0.6 * 4,

    #==> Transfer rule parameters
    'tau_D': 0.181,         'tau_F': 0.181,
    'lamb_D': 0.85,         'lamb_F': 0.85,
    'lamb_ss_D': 0.85,      'lamb_ss_F': 0.85,
    'phi_lamb_D': 0.1,      'phi_lamb_F': 0.1,

    #==> Default parameters
    'shock_def_D': 0.000,   'shock_def_F': 0.0,
    'def_rate_D':  0.000,   'def_rate_F':  0.0,   
    'def_curvature_D': 0.5, 'def_curvature_F': 0.5,
    'recovery_rate_D': 0.40,'recovery_rate_F': 0.40,

    #==> Aggregate targets
    'Y_D': 1.00,  'Y_F': 1.00,
    'N_D': 1.00,  'N_F': 1.00,
    'w_D': 0.65,  'w_F': 0.65,

    #==> Financial intermediary
    'f_D': 0.06,            'f_F': 0.06,
    'lambda_gk_D': 0.116,   'lambda_gk_F': 0.116,
    'ksi_D': 0.5,           'ksi_F': 0.5,
    'n_inter_D': 0.75 * 4,  'n_inter_F': 0.75 * 4,
    'theta_D': 4,           'theta_F': 4,

    #==> Production
    'alpha_D': 0.35,   'alpha_F': 0.35,
    'delta_D': 0.0125, 'delta_F': 0.0125,

    #==> Deposit grid
    'nZ_D': 19,       'nZ_F': 21,
    'nDep_D': 500,    'nDep_F': 500,
    'Depmax_D': 150,  'Depmax_F': 150,

    'rho_z_D': 0.9,   'rho_z_F': 0.97,
    'sigma_z_D': 0.5, 'sigma_z_F': 0.8,

    #==> Tobin's q
    'Q_D': 1.0,       'Q_F': 1.0,

    #==> Government bonds and fiscal-rule anchors
    'b_gov_D': 0.6 * 4,    'b_gov_F': 0.6 * 4,
    'b_gov_ss_D': 0.6 * 4, 'b_gov_ss_F': 0.6 * 4,

    #==> Trade
    'omega': 0.85,
    'epsilon_trade': 1.5,
    'p': 1.0,

    # ── Cross-border bond portfolio ────────────────────────────────────────────
    'phi_bF_D_ss': 0.15,  'phi_bD_F_ss': 0.20,
    'psi_bF_D':    0.05,  'psi_bD_F':    0.05,

    # ── NK frictions — D country ──────────────────────────────────────────────
    'mu_p_D':    1.0,    'mu_w_D':    1.0,
    'kappa_p_D': 0.10,   'kappa_w_D': 0.10,
    'phi_pi_D':  1.5,    'phi_y_D':   0.125,
    'rho_i_D':   0.8,    'r_star_D':  0.0065,
    'eps_m_D':   0.0,    'Y_ss_D':    1.0,
    'pi_D':  0.0,   'pi_w_D': 0.0,
    'i_D':   0.0065,
    'mc_D':  1.0,

    # ── NK frictions — F country ──────────────────────────────────────────────
    'mu_p_F':    1.0,    'mu_w_F':    1.0,
    'kappa_p_F': 0.10,   'kappa_w_F': 0.10,
    'phi_pi_F':  1.5,    'phi_y_F':   0.125,
    'rho_i_F':   0.8,    'r_star_F':  0.0062,
    'eps_m_F':   0.0,    'Y_ss_F':    1.0,
    'pi_F':  0.0,   'pi_w_F': 0.0,
    'i_F':   0.0062,
    'mc_F':  1.0,
}

calibration_start_D = {k: v for k, v in calibration_start.items() if k.endswith('_D')}
calibration_start_F = {k: v for k, v in calibration_start.items() if k.endswith('_F')}

calibration_hh_D = {**calibration_start_D, 'beta_D': 0.9705546368050272, 'div_D': 0.141500}
calibration_hh_F = {**calibration_start_F, 'beta_F': 0.9705546368050272, 'div_F': 0.141500}


In [ ]:
# Bonds calibration

_phi_bF_D = calibration_start['phi_bF_D_ss']   # D banks' target share in F bonds
_phi_bD_F = calibration_start['phi_bD_F_ss']   # F banks' target share in D bonds  (may differ)
_n_D      = calibration_start['n_inter_D']
_n_F      = calibration_start['n_inter_F']
_B_D      = calibration_start['B_supply_D']
_B_F      = calibration_start['B_supply_F']

b_F_D_ss  = _phi_bF_D * _n_D           # D banks' F bond holdings
b_D_F_ss  = _phi_bD_F * _n_F           # F banks' D bond holdings
b_D_D_ss  = _B_D - b_D_F_ss            # D banks' domestic bonds (clearing residual)
b_F_F_ss  = _B_F - b_F_D_ss            # F banks' domestic bonds (clearing residual)


excess_return_F_D_ss = calibration_start['rdep_F'] - calibration_start['rdep_D']
excess_return_D_F_ss = calibration_start['rdep_D'] - calibration_start['rdep_F']

calibration_start.update({
    'b_F_D': b_F_D_ss, 'b_D_F': b_D_F_ss,
    'b_D_D': b_D_D_ss, 'b_F_F': b_F_F_ss,
    'excess_return_F_D_ss': excess_return_F_D_ss,
    'excess_return_D_F_ss': excess_return_D_F_ss,
})



In [ ]:
from equations_D import (hh_init_D, hh_D, make_grids_D, income_D, hh_extended_D)

from equations_F import (hh_init_F, hh_F, make_grids_F, income_F, hh_extended_F)

### EQUATIONS

#### STEADY STATE EQUATIONS

In [ ]:
from equations_D import (
    smart_steady_D, market_clearing_D, steady_auxilliary_D,
    banker_div_D, sdf_D, sdf_ss_D, government_ss_D, labor_ss_D,
    government_default_D, interest_rates_D, bond_return_D,
    ces_price_D, import_demand_D,
)

from equations_F import (
    smart_steady_F, market_clearing_F, steady_auxilliary_F,
    banker_div_F, sdf_F, sdf_ss_F, government_ss_F, labor_ss_F,
    government_default_F, interest_rates_F, bond_return_F,
    ces_price_F, import_demand_F,
)

from equations_global import (
    trade_balance, global_bond_market, portfolio_adj_cost,
    domestic_bond_clearing, global_goods_mkt,
)


### SOLVING MODEL

#### STEADY STATE

In [ ]:
import copy

# ── Steady-state model ────────────────────────────────────────────────────────
ha = sj.create_model([
    sdf_ss_D, government_default_D, interest_rates_D, bond_return_D,
    sdf_ss_F, government_default_F, interest_rates_F, bond_return_F,

    hh_extended_D,
    smart_steady_D,      # uses rb_actual_D/F (post-haircut) for bond returns
    market_clearing_D,   # outputs goods_mkt_D — now the exchange rate target
    steady_auxilliary_D,
    banker_div_D,
    government_ss_D,     # uses rb_actual_D * b_gov_D (consistent with budget_residual_D)
    labor_ss_D,

    hh_extended_F,
    smart_steady_F,      # uses rb_actual_D for D-bond holdings by F banks
    market_clearing_F,
    steady_auxilliary_F,
    banker_div_F,
    government_ss_F,
    labor_ss_F,

    ces_price_D, import_demand_D,
    ces_price_F, import_demand_F,
    trade_balance,
    global_bond_market,
    global_goods_mkt,    # diagnostic: goods_mkt_D + p*goods_mkt_F ≈ 0 by Walras
], name="Simple HA Model 2 Country")

# ── Why goods_mkt_D is the exchange rate target ───────────────────────────────
# Walras identity (exact from HH + bank + government budgets):
#   goods_mkt_D ≡ −(NX_D + NFI_D) = −CA_D
# where NFI_D = p·rb_actual_F·b_F_D − rb_actual_D·b_D_F (net factor income).
# Old target NX_D = 0 forced a zero trade balance but left CA_D = NFI_D ≠ 0,
# making goods_mkt_D residual exactly equal to NFI_D ≈ 8e-4.
# Targeting goods_mkt_D = 0 directly enforces CA = 0 at SS without any
# lagged bond-position terms, giving clean exchange rate dynamics off-SS.
unknowns_ss = {
    'beta_D': 0.9923754668,
    'beta_F': 0.9879558013,
    'p':      0.998,        # lower than before: D needs small trade deficit to offset NFI
}
# goods_mkt_D pins p; deposit markets pin beta_D/F
targets_ss = ['deposit_mkt_D', 'deposit_mkt_F', 'goods_mkt_D']

ss = ha.solve_steady_state(calibration_start, unknowns_ss, targets_ss,
                           solver='broyden_custom')

# ── Post-solve patches ────────────────────────────────────────────────────────
p_ss = float(ss['p'])
phi_bF_D_actual = p_ss * float(calibration_start['b_F_D']) / float(calibration_start['n_inter_D'])
phi_bD_F_actual = (1.0/p_ss) * float(calibration_start['b_D_F']) / float(calibration_start['n_inter_F'])
calibration_start.update({'phi_bF_D_ss': phi_bF_D_actual, 'phi_bD_F_ss': phi_bD_F_actual})
ss.toplevel['phi_bF_D_ss'] = phi_bF_D_actual
ss.toplevel['phi_bD_F_ss'] = phi_bD_F_actual
ss.toplevel['b_F_D_res']   = 0.0
ss.toplevel['b_D_F_res']   = 0.0

ss.toplevel['pi_D']   = 0.0;  ss.toplevel['pi_F']   = 0.0
ss.toplevel['pi_w_D'] = 0.0;  ss.toplevel['pi_w_F'] = 0.0
ss.toplevel['i_D']    = float(ss['rdep_D'])
ss.toplevel['i_F']    = float(ss['rdep_F'])
ss.toplevel['mc_D']   = 1.0;  ss.toplevel['mc_F']   = 1.0
ss.toplevel['nkpc_p_res_D'] = 0.0;  ss.toplevel['nkpc_p_res_F'] = 0.0
ss.toplevel['nkpc_w_res_D'] = 0.0;  ss.toplevel['nkpc_w_res_F'] = 0.0
ss.toplevel['w_res_D']      = 0.0;  ss.toplevel['w_res_F']      = 0.0
ss.toplevel['taylor_res_D'] = 0.0;  ss.toplevel['taylor_res_F'] = 0.0
ss.toplevel['rdep_ante_D']  = float(ss['rdep_D'])
ss.toplevel['rdep_ante_F']  = float(ss['rdep_F'])

ss_D = ss;  ss_F = ss;  cali_D = ss;  cali_F = ss;  cali = ss
calibration = dict(ss)
ss_final = copy.deepcopy(ss)

# ── NFI / CA diagnostic ───────────────────────────────────────────────────────
NFI_D = (float(ss['p']) * float(ss['rb_actual_F']) * float(calibration_start['b_F_D'])
         - float(ss['rb_actual_D']) * float(calibration_start['b_D_F']))

print(f"beta_D        = {ss['beta_D']:.10f}   beta_F = {ss['beta_F']:.10f}")
print(f"p_ss          = {ss['p']:.8f}")
print(f"rb_D          = {ss['rb_D']:.6f}   rb_F        = {ss['rb_F']:.6f}")
print(f"rb_actual_D   = {ss['rb_actual_D']:.6f}   rb_actual_F = {ss['rb_actual_F']:.6f}")
print(f"NFI_D         = {NFI_D:.2e}   (p·rb_F·b_FD − rb_D·b_DF)")
print(f"NX_D          = {ss['NX_D']:.2e}   CA_D = NX_D+NFI_D = {float(ss['NX_D'])+NFI_D:.2e}")
print(f"goods_mkt_D   = {ss['goods_mkt_D']:.2e}   goods_mkt_F = {ss['goods_mkt_F']:.2e}")
print(f"global_goods  = {ss['global_goods_res']:.2e}   (should be ≈ 0 by Walras)")
print(f"tob_res=NX_D  = {ss['tob_res']:.2e}   (not targeted; ≈ −NFI_D at new SS)")
print(f"deposit_D     = {ss['deposit_mkt_D']:.2e}   deposit_F   = {ss['deposit_mkt_F']:.2e}")
print(f"ss_final defined ✓  (psi_bF_D={ss_final['psi_bF_D']:.3f}, psi_bD_F={ss_final['psi_bD_F']:.3f})")


#### OFF STEADY-STATE EQUATIONS

In [ ]:
from equations_D import (
    capital_adj_D, labor_D,
    intermediation_IC_D, bank_return_D, intermediation_P1_D,
    k_balance_sheet_D, intermediation_P2_D, banker_div_res_D,
    intermediation_P3_D, interest_rates_D, government_default_D,
    tax_rule_D, capital_producer_profit_D, budget_residual_D,
    ces_price_D, import_demand_D,
    bond_return_D,
    sdf_D,
    fisher_D, taylor_rule_D, pricing_D, wage_setting_D,
    labor_market_D,
)

from equations_F import (
    capital_adj_F, labor_F,
    intermediation_IC_F, bank_return_F, intermediation_P1_F,
    k_balance_sheet_F, intermediation_P2_F, banker_div_res_F,
    intermediation_P3_F, interest_rates_F, government_default_F,
    tax_rule_F, capital_producer_profit_F, budget_residual_F,
    ces_price_F, import_demand_F,
    bond_return_F,
    sdf_F,
    fisher_F, taylor_rule_F, pricing_F, wage_setting_F,
    labor_market_F,
)

from equations_global import (
    trade_balance, global_bond_market,
    domestic_bond_clearing,
    portfolio_adj_cost,
    global_goods_mkt,
)

print(calibration)


#### FULL MODEL

In [ ]:
financial_solved_D = combine([
    intermediation_IC_D, intermediation_P1_D,
]).solved(
    unknowns={'nu_D': float(cali_D['nu_D']), 'eta_D': float(cali_D['eta_D'])},
    targets=['nu_res_D', 'eta_res_D'],
    solver='broyden_custom'
)

financial_solved_F = combine([
    intermediation_IC_F, intermediation_P1_F,
]).solved(
    unknowns={'nu_F': float(cali_F['nu_F']), 'eta_F': float(cali_F['eta_F'])},
    targets=['nu_res_F', 'eta_res_F'],
    solver='broyden_custom'
)

ha_full = sj.create_model([
    # ── Country D ──────────────────────────────────────────────────────────────
    hh_extended_D,
    sdf_D,
    financial_solved_D,
    interest_rates_D,
    government_default_D,
    bond_return_D,
    bank_return_D,
    intermediation_P2_D,
    intermediation_P3_D,
    k_balance_sheet_D,
    capital_adj_D,
    tax_rule_D,
    capital_producer_profit_D,
    budget_residual_D,
    labor_D,
    pricing_D,
    wage_setting_D,
    labor_market_D,
    taylor_rule_D,
    fisher_D,
    banker_div_res_D,
    market_clearing_D,      # outputs goods_mkt_D (now exchange rate target)

    # ── Country F ──────────────────────────────────────────────────────────────
    hh_extended_F,
    sdf_F,
    financial_solved_F,
    interest_rates_F,
    government_default_F,
    bond_return_F,
    bank_return_F,
    intermediation_P2_F,
    intermediation_P3_F,
    k_balance_sheet_F,
    capital_adj_F,
    tax_rule_F,
    capital_producer_profit_F,
    budget_residual_F,
    labor_F,
    pricing_F,
    wage_setting_F,
    labor_market_F,
    taylor_rule_F,
    fisher_F,
    banker_div_res_F,
    market_clearing_F,

    # ── Global ─────────────────────────────────────────────────────────────────
    ces_price_D, import_demand_D,
    ces_price_F, import_demand_F,
    trade_balance,          # produces NX_D, NX_F, tob_res (=NX_D, not targeted)
    domestic_bond_clearing,
    portfolio_adj_cost,
    global_goods_mkt,       # diagnostic: global_goods_res = goods_mkt_D + p*goods_mkt_F
], name="Full 2-Country HANK-NK")

# ── 25 unknowns / 25 targets ──────────────────────────────────────────────────
# Exchange rate p is now pinned by goods_mkt_D = 0 (D goods market clears).
# By Walras' law (goods_mkt_D ≡ −CA_D from budget identities), this enforces
# CA = 0 at SS and goods market clearing at every period off-SS.
# global_goods_res = goods_mkt_D + p*goods_mkt_F ≈ 0 by Walras; check in IRFs.
unknowns_tp = [
    # D (12)
    'K_D', 'n_inter_D', 'div_D', 'I_D', 'Q_D', 'b_gov_D', 'Y_D', 'b_F_D',
    'pi_D', 'pi_w_D', 'w_D', 'i_D',
    # F (12)
    'K_F', 'n_inter_F', 'div_F', 'I_F', 'Q_F', 'b_gov_F', 'Y_F', 'b_D_F',
    'pi_F', 'pi_w_F', 'w_F', 'i_F',
    # Global (1)
    'p',
]
targets_tp = [
    # D (12)
    'deposit_mkt_D', 'K_res_D', 'n_inter_val_D', 'div_res_D',
    'capital_res_D', 'q_res_D', 'b_gov_res_D', 'b_F_D_res',
    'nkpc_p_res_D', 'nkpc_w_res_D', 'w_res_D', 'taylor_res_D',
    # F (12)
    'deposit_mkt_F', 'K_res_F', 'n_inter_val_F', 'div_res_F',
    'capital_res_F', 'q_res_F', 'b_gov_res_F', 'b_D_F_res',
    'nkpc_p_res_F', 'nkpc_w_res_F', 'w_res_F', 'taylor_res_F',
    # Global (1): goods market clears D → pins p; goods_mkt_F clears by Walras
    'goods_mkt_D',
]

T = 300
exogenous = ['Z_D', 'shock_def_D', 'Z_F', 'shock_def_F', 'eps_m_D', 'eps_m_F']

print(f"Unknowns ({len(unknowns_tp)}): {unknowns_tp}")
print(f"Targets  ({len(targets_tp)}): {targets_tp}")
print(f"\nComputing Jacobian for horizon T={T}...")
G_jac = ha_full.solve_jacobian(ss, unknowns_tp, targets_tp, exogenous, T=T)
print("Jacobian computed successfully.")


### IMPULSE RESPONSE FUNCTIONS

In [ ]:
def show_irfs(irfs_list, variables, labels=[" "], ylabel=r"Percentage points (dev. from ss)", T_plot=50, figsize=(18, 6)):
    if len(irfs_list) != len(labels):
        labels = [" "] * len(irfs_list)
    n_var = len(variables)
    fig, ax = plt.subplots(1, n_var, figsize=figsize, sharex=True)
    zeros = np.zeros(T_plot)
    for i in range(n_var):
        for j, irf in enumerate(irfs_list):
            try:
                data = irf[variables[i]][:T_plot]
            except KeyError:
                data = zeros
            ax[i].plot(data, label=labels[j])
        ax[i].set_title(variables[i])
        ax[i].set_xlabel(r"$t$")
        if i == 0:
            ax[i].set_ylabel(ylabel)
        ax[i].legend()
    plt.show()

In [ ]:
# ── Impulse Response Functions ────────────────────────────────────────────────

# TFP Shock
rho_Z_D = 0.8
dZ_D    = 0.01 * rho_Z_D ** np.arange(T)

print("Computing IRF: TFP shock...")
irfs_Z_D = ha_full.solve_impulse_linear(
    ss_final, unknowns_tp, targets_tp, {'Z_D': dZ_D}
)

# Default Shock
rho_def_D    = 0.8
dShock_def_D = 0.01 * rho_def_D ** np.arange(T)

print("Computing IRF: default shock...")
irfs_def_D = ha_full.solve_impulse_linear(
    ss_final, unknowns_tp, targets_tp, {'shock_def_D': dShock_def_D}
)

In [ ]:
real = ['Z_D', 'shock_def_D', 'def_rate_D', 'Y_D', 'C_D', 'I_D', 'K_D', 'Q_D', 'N_D', 'w_D', 'iota_D', 'mpk_D', 'cap_profit_D']

financial = [
    'theta_D', 'n_inter_D', 'rn_D', 'nu_D', 'eta_D',
    'rdep_D', 'rb_D', 'rb_actual_D',
    'b_gov_D', 'DEP_D', 'D_supply_D'
]

gov = ['def_rate_D', 'shock_def_D', 'b_gov_D', 'rb_D', 'rb_actual_D', 'TAX_D', 'lamb_D', 'G_D']

residuals = [
    'goods_mkt_D', 'deposit_mkt_D', 'K_res_D', 'n_inter_val_D',
    'div_res_D', 'capital_res_D', 'q_res_D', 'b_gov_res_D',
    'labor_mkt_res_D', 'nu_res_D', 'eta_res_D'
]

# ── IMPORTANT ────────────────────────────────────────────────────────────────
important = ['Y_D', 'Y_F', 'rb_D', 'rb_F', 'C_D', 'C_F']
show_irfs([irfs_Z_D],   important, ['Domestic TFP shock'])
show_irfs([irfs_def_D], important, ['Domestic default shock'])



In [ ]:
# ── Bond portfolio IRFs ───────────────────────────────────────────────────────
portfolio_vars = ['SDF_D','b_F_D', 'b_D_F', 'b_D_D', 'b_F_F', 'rb_D', 'rb_F']

show_irfs([irfs_Z_D],   portfolio_vars, ['Domestic TFP shock'])
show_irfs([irfs_def_D], portfolio_vars, ['Domestic default shock'])


In [ ]:
# ── Detailed (muted) ─────────────────────────────────────────────────────────
show_irfs([irfs_Z_D], real, ['TFP shock']) 
show_irfs([irfs_Z_D], financial, ['TFP shock'])
show_irfs([irfs_Z_D], residuals, ['TFP shock'])

show_irfs([irfs_def_D], real, ['Default shock'])
show_irfs([irfs_def_D], financial, ['Default shock']) 
show_irfs([irfs_def_D], gov, ['Default shock'])
show_irfs([irfs_def_D], residuals, ['Default shock'])

In [ ]:
# Variables requested: 
# 1. Discount factor (SDF_D)
# 2. Debt holdings (b_D_D)
# 3. rdep (rdep_D) 
# 4. rb (rb_D)
# 5. rb_actual (rb_actual_D)

requested_vars = ['SDF_D', 'b_D_D', 'rdep_D', 'rb_D', 'rb_actual_D']

# Assuming irfs_def_D and show_irfs are already defined in the notebook
show_irfs([irfs_def_D], requested_vars, ['Sovereign Default Shock'], T_plot=50)